# IsaacLab: Learning a Reward Function using Preference Comparisons

The preference comparisons algorithm learns a reward function by comparing trajectory segments to each other.

In [1]:
import random
from imitation.algorithms import preference_comparisons
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
from imitation.util.util import make_vec_env
from imitation.policies.base import FeedForward32Policy, NormalizeFeaturesExtractor
import gymnasium as gym
from stable_baselines3 import PPO
import numpy as np

rng = np.random.default_rng(0)

In [8]:
import argparse

from isaaclab.app import AppLauncher

# add argparse arguments
parser = argparse.ArgumentParser(description="Random agent for Isaac Lab environments.")
parser.add_argument(
    "--disable_fabric", action="store_true", default=False, help="Disable fabric and use USD I/O operations."
)
parser.add_argument("--num_envs", type=int, default="10", help="Number of environments to simulate.")
parser.add_argument("--task", type=str, default="Isaac-Ant-v0", help="Name of the task.")
# append AppLauncher cli args
AppLauncher.add_app_launcher_args(parser)
# parse the arguments
args_cli = parser.parse_args()



usage: ipykernel_launcher.py [-h] [--disable_fabric] [--num_envs NUM_ENVS]
                             [--task TASK] [--headless] [--livestream {0,1,2}]
                             [--enable_cameras] [--xr] [--device DEVICE]
                             [--verbose] [--info] [--experience EXPERIENCE]
                             [--rendering_mode {performance,quality,balanced,xr}]
                             [--kit_args KIT_ARGS]
ipykernel_launcher.py: error: unrecognized arguments: -f /home/lingheng/snap/code/217/.local/share/jupyter/runtime/kernel-a20cb063-c39f-4c75-b379-e4fb1be616ae.json


SystemExit: 2

In [2]:
import argparse

from isaaclab.app import AppLauncher

# add argparse arguments
parser = argparse.ArgumentParser(description="Random agent for Isaac Lab environments.")
parser.add_argument(
    "--disable_fabric", action="store_true", default=False, help="Disable fabric and use USD I/O operations."
)
parser.add_argument("--num_envs", type=int, default="10", help="Number of environments to simulate.")
parser.add_argument("--task", type=str, default="Isaac-Ant-v0", help="Name of the task.")
# append AppLauncher cli args
AppLauncher.add_app_launcher_args(parser)
# # ********** original ******************
# # parse the arguments
# args_cli = parser.parse_args()
# # ********** original ******************


# Pass a custom list to simulate command-line input
args_cli = parser.parse_args(['--num_envs', '10', 
                              '--task', 'Isaac-Ant-v0', '--headless'])


[Warning] [simulation_app] Interactive python shell detected but ISAAC_JUPYTER_KERNEL was not set. Problems with asyncio may occur
[Warning] [simulation_app] Please use Isaac Sim Python 3 kernel instead of the default Python 3 Kernel


In [3]:
args_cli

Namespace(disable_fabric=False, num_envs=10, task='Isaac-Ant-v0', headless=True, livestream=-1, enable_cameras=False, xr=False, device='cuda:0', cpu=False, verbose=False, info=False, experience='', rendering_mode=None, kit_args='')

In [4]:
# launch omniverse app
app_launcher = AppLauncher(args_cli)
simulation_app = app_launcher.app

"""Rest everything follows."""

import gymnasium as gym
import torch

import isaaclab_tasks  # noqa: F401
from isaaclab_tasks.utils import parse_env_cfg

import isaaclab_csiro_hri.tasks  # noqa: F401

# create environment configuration
env_cfg = parse_env_cfg(
    args_cli.task, device=args_cli.device, num_envs=args_cli.num_envs, use_fabric=not args_cli.disable_fabric
)
# create environment
env = gym.make(args_cli.task, cfg=env_cfg)

[INFO][AppLauncher]: Using device: cuda:0
[INFO][AppLauncher]: Loading experience file: /home/lingheng/SynologyDrive/Project_Ongoing_Now_2024-07-26_CSIRO-Science-Digital/Project_Ongoing_Now_2024-08-26_CSIRO-Science-Digital-MSI-Titan-Backup/IsaacLab_Stack_3/hri-ppl/IsaacLab/apps/isaaclab.python.headless.kit
Loading user config located at: '/home/lingheng/isaacsim/4.5.0/kit/data/Kit/Isaac-Sim/4.5/user.config.json'
[Info] [carb] Logging to file: /home/lingheng/isaacsim/4.5.0/kit/logs/Kit/Isaac-Sim/4.5/kit_20260105_174002.log
2026-01-05 06:40:02 s] [Warning] [omni.kit.app.plugin] No crash reporter present, dumps uploading isn't available.
2026-01-05 06:40:02 s] [Warning] [omni.ext.plugin] [ext: rendering_modes] Extensions config 'extension.toml' doesn't exist '/home/lingheng/SynologyDrive/Project_Ongoing_Now_2024-07-26_CSIRO-Science-Digital/Project_Ongoing_Now_2024-08-26_CSIRO-Science-Digital-MSI-Titan-Backup/IsaacLab_Stack_3/hri-ppl/IsaacLab/apps/rendering_modes' or '/home/lingheng/Synolo

AttributeError: '_UnixSelectorEventLoop' object has no attribute '_old_agen_hooks'

2026-01-05 06:40:06 [4,074ms] [Warning] [gpu.foundation.plugin] Skipping unsupported non-NVIDIA GPU: Intel(R) Graphics (RPL-S)
2026-01-05 06:40:06 [4,074ms] [Warning] [gpu.foundation.plugin] Skipping unsupported non-NVIDIA GPU: Intel(R) Graphics (RPL-S)

|---------------------------------------------------------------------------------------------|
| Driver Version: 570.195.03    | Graphics API: Vulkan
|=============================================================================================|
| GPU | Name                             | Active | LDA | GPU Memory | Vendor-ID | LUID       |
|     |                                  |        |     |            | Device-ID | UUID       |
|     |                                  |        |     |            | Bus-ID    |            |
|---------------------------------------------------------------------------------------------|
| 0   | NVIDIA GeForce RTX 4090 Laptop.. | Yes: 0 |     | 16376   MB | 10de      | 0          |
|     |           

2026-01-05 06:40:07 [4,727ms] [Error] [asyncio] Exception in callback <TaskStepMethWrapper object at 0x7c4f4159dd80>()
handle: <Handle <TaskStepMethWrapper object at 0x7c4f4159dd80>()>
Traceback (most recent call last):
  File "/home/lingheng/anaconda3/envs/env_isaaclab/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: Cannot enter into task <Task pending name='Task-4' coro=<UsdExtension.__init_stage_event() running at /home/lingheng/isaacsim/4.5.0/extscache/omni.usd-1.12.4+d02c707b.lx64.r.cp310/omni/usd/_impl/__init__.py:29>> while another task <Task pending name='Task-1' coro=<Kernel.dispatch_queue() running at /home/lingheng/anaconda3/envs/env_isaaclab/lib/python3.10/site-packages/ipykernel/kernelbase.py:510> cb=[_wrap_awaitable.<locals>.<lambda>() at /home/lingheng/IsaacLab_Stack_3/hri-ppl/IsaacLab/_isaac_sim/exts/omni.isaac.core_archive/pip_prebundle/tornado/gen.py:851, IOLoop.add_future.<locals>.<lambda>() at /hom

[INFO]: Parsing configuration from: isaaclab_tasks.manager_based.classic.ant.ant_env_cfg:AntEnvCfg
2026-01-05 06:40:07 [4,939ms] [Warning] [isaaclab.envs.manager_based_env] Seed not set for the environment. The environment creation may not be deterministic.
[INFO]: Base environment:
	Environment device    : cuda:0
	Environment seed      : None
	Physics step-size     : 0.008333333333333333
	Rendering step-size   : 0.016666666666666666
	Environment step-size : 0.016666666666666666
[INFO]: Time taken for scene creation : 1.254786 seconds
[INFO]: Scene manager:  <class InteractiveScene>
	Number of environments: 10
	Environment spacing   : 5.0
	Source prim name      : /World/envs/env_0
	Global prim paths     : ['/World/ground']
	Replicate physics     : True
[INFO]: Starting the simulation. This may take a few seconds. Please wait...
[INFO]: Time taken for simulation start : 0.449823 seconds
[INFO] Command Manager:  <CommandManager> contains 0 active terms.
+------------------------+
|  Acti

[INFO] Observation Manager: <ObservationManager> contains 1 groups.
+-----------------------------------------------------------+
| Active Observation Terms in Group: 'policy' (shape: (60,)) |
+-----------+-----------------------------------+-----------+
|   Index   | Name                              |   Shape   |
+-----------+-----------------------------------+-----------+
|     0     | base_height                       |    (1,)   |
|     1     | base_lin_vel                      |    (3,)   |
|     2     | base_ang_vel                      |    (3,)   |
|     3     | base_yaw_roll                     |    (2,)   |
|     4     | base_angle_to_target              |    (1,)   |
|     5     | base_up_proj                      |    (1,)   |
|     6     | base_heading_proj                 |    (1,)   |
|     7     | joint_pos_norm                    |    (8,)   |
|     8     | joint_vel_rel                     |    (8,)   |
|     9     | feet_body_forces                  |   (24,)   |
|

In [ ]:
# create environment configuration
env_cfg = parse_env_cfg(
    args_cli.task, device=args_cli.device, num_envs=args_cli.num_envs, use_fabric=not args_cli.disable_fabric
)
# create environment
env = gym.make(args_cli.task, cfg=env_cfg)

In [ ]:
# print info (this is vectorized environment)
print(f"[INFO]: Gym observation space: {env.observation_space}")
print(f"[INFO]: Gym action space: {env.action_space}")